# Aprire la scatola nera

Il codice del capitolo [«Aprire la scatola nera»](https://book.paithon.it/main/Interpretabilita/overview.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy scikit-learn torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Aprire la scatola nera

[Leggi la pagina](https://book.paithon.it/main/Interpretabilita/overview.html)


### Un modello che si spiega da sé


In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, export_text

# Un modello intrinsecamente interpretabile: un albero volutamente basso
iris = load_iris()
X, y = iris.data, iris.target
albero = DecisionTreeClassifier(max_depth=2, random_state=0)
albero.fit(X, y)

# Tutta la "logica" del modello è leggibile come una ricetta di if-then
print(export_text(albero, feature_names=list(iris.feature_names)))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# dieci gruppi: nove per imparare, uno per l'esame, e si gira dieci volte
albero_alto = DecisionTreeClassifier(max_depth=3, random_state=0)
foresta = RandomForestClassifier(n_estimators=300, random_state=0)
for nome, m in [("alberello", albero), ("alberello più alto", albero_alto),
                ("foresta casuale", foresta)]:
    print(f"{nome:18} {cross_val_score(m, X, y, cv=10).mean():.1%}")

### Una spiegazione può convincere ed essere falsa


In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(0)
n = 4000
reddito = rng.normal(size=n)
spesa = reddito + 0.1 * rng.normal(size=n)          # quasi una copia del reddito
eta = rng.normal(size=n)
y = (reddito + eta + rng.normal(size=n) > 0).astype(int)
X = np.column_stack([reddito, spesa, eta])
X_tr, X_te, y_tr, y_te = X[:2000], X[2000:], y[:2000], y[2000:]
nomi = ["reddito", "spesa", "età"]

def importanza(modello, colonne, j, ripetizioni=20):
    """Quanto cala l'accuratezza rimescolando la colonna j, in media su più rimescolamenti
    (zero se il modello non la usa)."""
    if j not in colonne:
        return 0.0
    base, cali = modello.score(X_te[:, colonne], y_te), []
    for r in range(ripetizioni):
        A = X_te.copy()
        A[:, j] = np.random.default_rng(r).permutation(A[:, j])
        cali.append(base - modello.score(A[:, colonne], y_te))
    return float(np.mean(cali))

modelli = {"logistica su reddito ed età": (LogisticRegression(), [0, 2]),
           "logistica su spesa ed età": (LogisticRegression(), [1, 2]),
           "boosting su tutte e tre": (GradientBoostingClassifier(random_state=0), [0, 1, 2])}
for nome, (m, colonne) in modelli.items():
    m.fit(X_tr[:, colonne], y_tr)
    imp = ", ".join(f"{nomi[j]} {importanza(m, colonne, j):.3f}" for j in (0, 1))
    print(f"{nome:28}: accuratezza {m.score(X_te[:, colonne], y_te):.4f}; importanza {imp}")

# il boosting regolato: la migliore di sei combinazioni di profondità e passo
migliore = max(GradientBoostingClassifier(max_depth=d, learning_rate=lr, random_state=0)
               .fit(X_tr, y_tr).score(X_te, y_te)
               for d in (1, 2, 3) for lr in (0.03, 0.1))
print(f"boosting regolato, il migliore di sei: accuratezza {migliore:.4f}")

## Modelli trasparenti e importanza delle feature

[Leggi la pagina](https://book.paithon.it/main/Interpretabilita/modelli-trasparenti-e-importanza.html)


### Modelli trasparenti per costruzione


In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LassoCV, LinearRegression
from sklearn.metrics import r2_score

rng = np.random.default_rng(0)
n, nomi = 3000, ["età", "reddito", "anzianità", "rate", "figli"]
X = rng.uniform(0, 1, size=(n, 5))
# la regola nascosta: un'interazione a soglia, più un effetto lineare
y = 3.0 * ((X[:, 0] > 0.6) & (X[:, 1] < 0.4)) + 2.0 * X[:, 2] + rng.normal(0, 0.3, n)
X_tr, X_te, y_tr, y_te = X[:2000], X[2000:], y[:2000], y[2000:]

foresta = GradientBoostingRegressor(n_estimators=50, max_depth=2, learning_rate=0.1,
                                    subsample=0.5, random_state=0).fit(X_tr, y_tr)

def regole(albero):
    """Ogni nodo che non è la radice è una regola: le condizioni del percorso."""
    t, trovate = albero.tree_, []
    def scendi(nodo, condizioni):
        if t.children_left[nodo] == -1:
            return
        # soglie arrotondate al centesimo e condizioni in ordine fisso:
        # la stessa regola trovata da alberi diversi diventa una sola
        j, s = t.feature[nodo], round(float(t.threshold[nodo]), 2)
        for figlio, segno in ((t.children_left[nodo], "<="), (t.children_right[nodo], ">")):
            nuove = condizioni + ((j, segno, s),)
            trovate.append(tuple(sorted(nuove)))
            scendi(figlio, nuove)
    scendi(0, ())
    return trovate

tutte = sorted({r for stima in foresta.estimators_[:, 0] for r in regole(stima)})
def vale(regola, X):
    ok = np.ones(len(X), bool)
    for j, segno, s in regola:
        ok &= (X[:, j] <= s) if segno == "<=" else (X[:, j] > s)
    return ok.astype(float)
# i termini lineari, riscalati come nel lavoro originale: 0,4 / deviazione standard
# (la winsorizzazione qui si omette: dati uniformi in [0, 1] non hanno code)
scala = 0.4 / X_tr.std(axis=0)
def colonne(A):
    return np.column_stack([vale(r, A) for r in tutte] + [A * scala])
R_tr, R_te = colonne(X_tr), colonne(X_te)
lasso = LassoCV(cv=5).fit(R_tr, y_tr)

# attivi i coefficienti non nulli; secondo il processore alcuni zeri escono
# come residui di arrotondamento dell'ordine di 1e-17, e non vanno contati
attive = np.flatnonzero(np.abs(lasso.coef_) > 1e-10)
print(f"regole candidate: {len(tutte)}, termini tenuti dal Lasso: {len(attive)}")
for nome, modello, A in [("lineare", LinearRegression().fit(X_tr, y_tr), X_te),
                         ("boosting", foresta, X_te),
                         ("RuleFit", lasso, R_te)]:
    print(f"R^2 sul test, {nome:8}: {r2_score(y_te, modello.predict(A)):.2f}")
# il tetto: la regola vera senza il rumore, che nessun modello può battere
vera = 3.0 * ((X_te[:, 0] > 0.6) & (X_te[:, 1] < 0.4)) + 2.0 * X_te[:, 2]
print(f"R^2 sul test, la regola vera: {r2_score(y_te, vera):.2f}")
testo = lambda r: " E ".join(f"{nomi[j]} {s_} {v:.2f}" for j, s_, v in r)
pesi = [(abs(lasso.coef_[k]) * R_tr[:, k].std(), k) for k in attive]
tot = sum(p for p, _ in pesi)
print(f"peso dei tre termini più importanti: {sum(p for p, _ in sorted(pesi, reverse=True)[:3]) / tot:.0%}")
for _, k in sorted(pesi, reverse=True)[:3]:
    if k < len(tutte):
        print(f"{lasso.coef_[k]:+.2f}  SE {testo(tutte[k])}")
    else:                                  # riportato all'unità della colonna
        j = k - len(tutte)
        print(f"{lasso.coef_[k] * scala[j]:+.2f}  per ogni unità di {nomi[j]}")
# l'anzianità non sta solo nel suo termine lineare: di quanto sale in tutto la
# previsione per un'unità in più (da 0,05 a 0,95, sui dati di prova)
alto, basso = X_te.copy(), X_te.copy()
alto[:, 2], basso[:, 2] = 0.95, 0.05
salto = lasso.predict(colonne(alto)) - lasso.predict(colonne(basso))
salita = salto.mean() / 0.9
sola = sum(1 for k in attive
           if k < len(tutte) and all(j == 2 for j, _, _ in tutte[k]))
print(f"regole sulla sola anzianità: {sola};",
      f"in tutto, per unità: {salita:+.2f}")

### In pratica: rimescolamento contro impurità


In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

dati = load_diabetes()
X, y, nomi = dati.data, dati.target, list(dati.feature_names)

# Due colonne di puro rumore, scorrelate dal target: una continua e una binaria.
# Non valgono niente ne l'una ne l'altra: servono da metro per le due misure.
rng = np.random.default_rng(0)
X = np.column_stack([X, rng.normal(size=len(y)), rng.integers(0, 2, size=len(y))])
nomi += ["rumore_cont", "rumore_bin"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

rf = RandomForestRegressor(n_estimators=300, random_state=0)
rf.fit(X_tr, y_tr)
print("R^2 sul test:", round(rf.score(X_te, y_te), 3))  # -> 0.315

# Importanza da permutazione, misurata sul TEST (10 mescolamenti per feature)
pi = permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=0)

print("feature      (impurita)   perm-import   valori distinti")
for i in np.argsort(rf.feature_importances_)[::-1]:   # dalla piu alta per la MDI
    print(f"{nomi[i]:>11}   {rf.feature_importances_[i]:.4f}      "
          f"{pi.importances_mean[i]:+.3f} +/- {pi.importances_std[i]:.3f}"
          f"   {len(np.unique(X_tr[:, i])):5d}")

### Spiegare con gli esempi: prototipi e critiche


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
# due gruppi grandi e un gruppetto raro, che una sintesi per medie rischia di perdere
grande_a = rng.normal([0, 0], 0.5, size=(200, 2))
grande_b = rng.normal([4, 0], 0.5, size=(200, 2))
raro = rng.normal([2, 3], 0.3, size=(20, 2))
X = np.vstack([grande_a, grande_b, raro])
gruppo = np.array(["a"] * 200 + ["b"] * 200 + ["raro"] * 20)

def nucleo(A, B, larghezza=1.0):
    d2 = ((A[:, None, :] - B[None, :, :]) ** 2).sum(-1)
    return np.exp(-d2 / (2 * larghezza ** 2))

K = nucleo(X, X)
media_dati = K.mean(axis=1)                  # quanto ogni punto somiglia ai dati, in media

def mmd2(scelti):
    """MMD al quadrato fra i dati e i soli prototipi scelti (a meno della costante dei dati)."""
    S = np.array(scelti)
    return K[np.ix_(S, S)].mean() - 2 * media_dati[S].mean()

prototipi = []
for _ in range(10):                          # scelta avida: il prototipo che abbassa di più l'MMD
    candidati = [i for i in range(len(X)) if i not in prototipi]
    prototipi.append(min(candidati, key=lambda i: mmd2(prototipi + [i])))
print("prototipi per gruppo:", {g: int((gruppo[prototipi] == g).sum()) for g in ("a", "b", "raro")})

# la funzione testimone: dove i dati sono più fitti dei prototipi (positiva) o meno (negativa)
testimone = media_dati - K[:, prototipi].mean(axis=1)
critiche = np.argsort(-testimone)[:3]         # dove i dati sono più fitti di quanto i prototipi dicano
print("critiche:", [str(gruppo[i]) for i in critiche])
print(f"testimone più alta, nel gruppo raro {testimone[gruppo == 'raro'].max():.3f},"
      f" fuori {testimone[gruppo != 'raro'].max():.3f}")
distanza = max(np.linalg.norm(X[i] - X[j]) for i in critiche for j in critiche)
print(f"distanza massima fra le tre critiche: {distanza:.2f}")

# il metodo originale: la testimone in valore assoluto più il log-determinante
# del nucleo sulle critiche scelte, che le vuole lontane fra loro
def punteggio(scelte):
    return (np.abs(testimone[scelte]).sum()
            + np.linalg.slogdet(K[np.ix_(scelte, scelte)])[1])

fuori = [i for i in range(len(X)) if i not in prototipi]
diverse = []
for _ in range(3):                           # di nuovo una alla volta
    diverse.append(max((i for i in fuori if i not in diverse),
                       key=lambda i: punteggio(diverse + [i])))
print("critiche col log-determinante:",
      [f"{gruppo[i]} {testimone[i]:+.3f}" for i in diverse])

## Spiegazioni locali: LIME, SHAP e controfattuali

[Leggi la pagina](https://book.paithon.it/main/Interpretabilita/spiegazioni-locali.html)


### Uno o molti: la diversità dei controfattuali


In [ ]:
import numpy as np

# un modello lineare per il prestito: reddito (migliaia di euro l'anno),
# rata dei debiti (centinaia di euro al mese), anni nell'impiego attuale
nomi = ["reddito", "rata", "anni di lavoro"]
w = np.array([1 / 6, -1 / 3, 1 / 6])
x0 = np.array([24.0, 6.0, 2.0])
b = -1.0 - w @ x0                        # la richiedente sta a logit -1: negato
mad = np.array([8.0, 3.0, 4.0])          # deviazione assoluta mediana di ogni voce


def costo(c, x):
    """Distanza L1 pesata con la MAD, quella di Wachter e colleghi."""
    return np.sum(np.abs(c - x) / mad)


# con un modello lineare il controfattuale più vicino muove una voce sola:
# per ciascuna, quanto va spostata perché il logit arrivi a zero
leve = []
for j in range(3):
    c = x0.copy()
    c[j] += -(w @ x0 + b) / w[j]
    leve.append(c)
    print(f"solo {nomi[j]:15s} -> {c[j]:5.1f}   costo {costo(c, x0):.2f}")
piu_vicino = int(np.argmin([costo(c, x0) for c in leve]))
print("il controfattuale più vicino muove:", nomi[piu_vicino])


def diversita(insieme):
    """Il determinante di K, con K_ij = 1 / (1 + distanza fra c_i e c_j)."""
    K = np.array([[1 / (1 + costo(ci, cj)) for cj in insieme] for ci in insieme])
    return np.linalg.det(K)


def perdita(insieme, l1=0.5, l2=1.0):
    """L'obiettivo di DiCE: cerniera sul logit, distanza media, meno la diversità."""
    cerniera = np.mean([max(0.0, 1 - (w @ c + b)) for c in insieme])
    distanza = np.mean([costo(c, x0) for c in insieme])
    return cerniera + l1 * distanza - l2 * diversita(insieme)


varianti = [x0 + [d, 0, 0] for d in (6.0, 6.5, 7.0)]   # tre volte la stessa leva
for nome, insieme in [("tre leve diverse", leve), ("tre volte il reddito", varianti)]:
    print(f"{nome:21s} distanza media {np.mean([costo(c, x0) for c in insieme]):.4f}, "
          f"diversità {diversita(insieme):.4f}, perdita {perdita(insieme):.4f}")

### In pratica: i valori di Shapley calcolati da zero


In [ ]:
import itertools
from math import factorial
import numpy as np

# Un modello giocattolo con un'interazione: la feature 2 "conta" solo con la 0
def f(x):
    return x[0] + 2.0 * x[1] + x[0] * x[2]

# istanza da spiegare e riferimento (baseline) su cui "spegnere" le feature assenti
x = np.array([1.0, 1.0, 1.0])
r = np.array([0.0, 0.0, 0.0])
n = len(x)

# valore della coalizione S: le feature in S prendono il valore di x, le altre di r
def v(S):
    z = r.copy()
    for i in S:
        z[i] = x[i]
    return f(z)

# valori di Shapley per forza bruta: media dei contributi marginali su TUTTI gli ordini
phi = np.zeros(n)
for perm in itertools.permutations(range(n)):
    S = []
    for i in perm:
        prima = v(S)            # coalizione prima di aggiungere i
        S = S + [i]
        dopo = v(S)             # coalizione dopo aver aggiunto i
        phi[i] += dopo - prima  # contributo marginale di i in questo ordine
phi /= factorial(n)             # media sugli n! ordini

print("valori di Shapley:", np.round(phi, 3))
print("somma dei phi:     ", round(float(phi.sum()), 3))
print("f(x) - f(base):    ", round(float(f(x) - f(r)), 3))  # assioma di efficienza

## Dentro le reti profonde: attribuzione e interpretabilità meccanicistica

[Leggi la pagina](https://book.paithon.it/main/Interpretabilita/attribuzione-e-meccanicistica.html)


### Integrated Gradients coi numeri: un esempio eseguibile


In [ ]:
import numpy as np

# funzione giocattolo che satura: f(x) = tanh(w . x)
w = np.array([2.0, -1.0])

def f(x):
    return np.tanh(w @ x)

def grad_f(x):
    z = w @ x
    return (1.0 - np.tanh(z) ** 2) * w   # regola della catena

x = np.array([2.0, 1.0])     # input da spiegare
baseline = np.zeros(2)        # baseline neutra (lo "zero")

# gradiente grezzo nel solo punto x: saturo, quasi nullo -> saliency cieca
print("gradiente in x :", np.round(grad_f(x), 4))       # [ 0.0197 -0.0099]

# Integrated Gradients: media dei gradienti lungo il cammino baseline -> x
m = 200
alphas = (np.arange(1, m + 1) - 0.5) / m   # punti medi delle m tappe
grad_medio = np.zeros(2)
for a in alphas:
    grad_medio += grad_f(baseline + a * (x - baseline))
grad_medio /= m
ig = (x - baseline) * grad_medio
print("attribuzioni IG:", np.round(ig, 4))              # [ 1.3267 -0.3317]

# assioma di completezza: la somma delle attribuzioni = f(x) - f(baseline)
print("somma IG       :", round(ig.sum(), 4))           # 0.9951
print("f(x) - f(base) :", round(f(x) - f(baseline), 4)) # 0.9951

### Tre famiglie sullo stesso neurone: gradiente, propagazione, perturbazione


In [ ]:
import numpy as np

w = np.array([2.0, -1.0])
x = np.array([2.0, 1.0])
f = lambda v: np.tanh(w @ v)
z, zi = w @ x, w * x                    # la somma pesata e i suoi due pezzi
delta = f(x) - f(np.zeros(2))           # quanto l'uscita si allontana dalla baseline

grad_per_ingresso = x * (1 - np.tanh(z) ** 2) * w
occlusione = np.array([f(x) - f(np.where(np.arange(2) == i, 0.0, x))
                       for i in range(2)])     # si azzera una variabile per volta
lrp = zi / z * f(x)                             # regola epsilon, con epsilon -> 0
deeplift = (f(x) - f(np.zeros(2))) / (z - 0.0) * zi   # rapporto incrementale
m = 200
alphas = (np.arange(1, m + 1) - 0.5) / m
ig = x * np.mean([(1 - np.tanh(a * z) ** 2) * w for a in alphas], axis=0)

print("somma pesata:", z, "  senza la prima:", w[1] * x[1],
      "  senza la seconda:", w[0] * x[0])
for nome, a in [("gradiente per ingresso", grad_per_ingresso),
                ("occlusione", occlusione), ("LRP", lrp),
                ("DeepLIFT", deeplift), ("gradienti integrati", ig)]:
    print(f"{nome:23s} {np.round(a, 4)}  somma {a.sum():.4f}")
print(f"{'f(x) - f(0)':23s} {delta:.4f}")

### Uno sketch di Grad-CAM in PyTorch


In [ ]:
import torch
import torch.nn.functional as F
from torchvision import models

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).eval()
target = model.layer4[-1]                # ultimo blocco: uscita post-residuo

att, grad = {}, {}
target.register_forward_hook(lambda m, i, o: att.__setitem__("v", o.detach()))
target.register_full_backward_hook(
    lambda m, gi, go: grad.__setitem__("v", go[0].detach())
)

x = torch.randn(1, 3, 224, 224)          # immagine gia pre-processata
logit = model(x)                          # (1, 1000)
classe = logit.argmax(dim=1)              # classe predetta
model.zero_grad()
logit[0, classe].backward()               # gradiente della sola classe scelta

A = att["v"]                              # attivazioni  (1, C, h, w)
dY = grad["v"]                            # gradienti    (1, C, h, w)
alpha = dY.mean(dim=(2, 3), keepdim=True)  # peso per canale (global avg pool)
heatmap = F.relu((alpha * A).sum(dim=1))   # (1, h, w), solo contributi positivi
heatmap = heatmap / (heatmap.max() + 1e-8) # normalizzata in [0, 1]
# heatmap va poi sovracampionata a 224x224 e sovrapposta all'immagine

print("attivazioni:", tuple(A.shape))
print("gradienti  :", tuple(dY.shape))
print("heatmap    :", tuple(heatmap.shape))